In [1]:
import requests, csv
from bs4 import BeautifulSoup
import re
import pandas as pd
import numpy as np
from collections import OrderedDict
import tqdm
from urllib.request import Request, urlopen
import os
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

In [2]:
from sqlalchemy import create_engine
import streamlit as st
import pandas as pd
import psycopg2
import time

In [3]:
def get_connection(): 
    try: 
        return psycopg2.connect( database="leo_cinema_db", 
                                user='postgres', 
                                password='postgres', 
                                host='localhost', 
                                port='5432', options="-c client_encoding=latin1" ) 
    except Exception as e: 
        st.error(f"Erreur connexion BD : {e}") 
        return None


def load_movies(): 
    conn = get_connection() 
    if conn is None: 
        return pd.DataFrame() 
    
    df = pd.read_sql("SELECT movie_id,french_title,original_title, year FROM movies", conn) 
    conn.close() 
    return df


def load_users():
    try:
        return pd.read_csv("db_backup/user_list_db.csv", sep=";")
    except Exception as e:
        st.error(f"Erreur chargement CSV : {e}")
        return pd.DataFrame()

In [4]:
movies = load_movies()
users = load_users()

C:\Users\lione\AppData\Local\Temp\ipykernel_21916\1586138590.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT movie_id,french_title,original_title, year FROM movies", conn)


In [8]:
movies[["french_title", "original_title"]] = movies[["french_title", "original_title"]].astype(str)
movies["search_fr"] = movies["french_title"].str.lower()
movies["search_or"] = movies["original_title"].str.lower()

In [9]:
movies

,movie_id,french_title,original_title,year,search_fr,search_or
0,tt0331811,11:14,11:14,2003,11:14,11:14
1,tt0416449,300,300,2006,300,300
2,tt8579674,1917,1917,2019,1917,1917
3,tt0078723,1941,1941,1979,1941,1941
4,tt0087803,1984,Nineteen Eighty-Four,1984,1984,nineteen eighty-four
...,...,...,...,...,...,...
89780,tt36749000,La poupée,La poupée,2025,la poupée,la poupée
89781,tt37741089,L'arnaqueuse,L'arnaqueuse,2026,l'arnaqueuse,l'arnaqueuse
89782,tt33515714,À voix basse,À voix basse,2026,à voix basse,à voix basse
89783,tt37660506,Les Fleurs du manguier,Harà watan,2025,les fleurs du manguier,harà watan


In [13]:
def load_actors(name): 
    conn = get_connection() 
    if conn is None: 
        return pd.DataFrame() 

    # 1. Normalisation : on s'assure d'obtenir une liste de noms propres
    if isinstance(name, list):
        names = [val.strip().lower() for val in name if isinstance(val, str) and val.strip()]
    elif isinstance(name, str):
        names = [val.strip().lower() for val in name.split(",") if val.strip()]
    else:
        names = []

    # 2. Exécution de la requête
    if names:
        # Conditions LIKE dynamiques : LOWER(a.name) LIKE %s OR ...
        like_conditions = " OR ".join(["LOWER(a.name) LIKE %s" for _ in names])
        
        # Correction ici : on utilise 'n' pour chaque élément de 'names' (et non la variable globale 'name')
        params = [f"%{n}%" for n in names]
        
        # Requête SQL avec GROUP BY et HAVING COUNT(DISTINCT)
        query = f"""
            SELECT m.movie_id, m.french_title, m.original_title, m.year
            FROM movies m
            JOIN movie_actor ma ON m.movie_id = ma.movie_id
            JOIN actors a ON ma.actor_id = a.actor_id
            WHERE {like_conditions}
            GROUP BY m.movie_id, m.french_title, m.original_title, m.year
            HAVING COUNT(DISTINCT a.actor_id) >= %s
        """
        
        # Ajout du nombre d'acteurs requis pour l'intersection (HAVING)
        params.append(len(names))
        
        df = pd.read_sql(query, conn, params=params)

    else:
        df = pd.DataFrame()

    conn.close() 
    return df

In [36]:
act = load_actors(["Tom Cruise","Simon Pegg"])

C:\Users\lione\AppData\Local\Temp\ipykernel_21916\4187555454.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn, params=params)


In [28]:
query = "Wide"

In [37]:
act

,movie_id,french_title,original_title,year
0,tt0317919,Mission: Impossible III,Mission: Impossible III,2006
1,tt1229238,Mission: Impossible - Protocole fantôme,Mission: Impossible - Ghost Protocol,2011
2,tt2381249,Mission: Impossible - Rogue Nation,Mission: Impossible - Rogue Nation,2015
3,tt4912910,Mission: Impossible - Fallout,Mission: Impossible - Fallout,2018
4,tt9603208,Mission: Impossible - The Final Reckoning,Mission: Impossible - The Final Reckoning,2025
5,tt9603212,Mission: Impossible: Dead Reckoning Partie 1,Mission: Impossible - Dead Reckoning Part One,2023


In [27]:
clean_movie = ["protocole fantôme", "rogue nation"]

In [39]:
movie = "test"

In [40]:
movie.split(",")

['test']

In [23]:
act[act["french_title"].str.lower().str.contains(clean_movie, na=False) | act["original_title"].str.lower().str.contains(clean_movie, na=False)]

,movie_id,french_title,original_title,year
17,tt0117060,Mission: Impossible,Mission: Impossible,1996
19,tt0120755,Mission: Impossible 2,Mission: Impossible II,2000
23,tt0317919,Mission: Impossible III,Mission: Impossible III,2006
31,tt1229238,Mission: Impossible - Protocole fantôme,Mission: Impossible - Ghost Protocol,2011
37,tt2381249,Mission: Impossible - Rogue Nation,Mission: Impossible - Rogue Nation,2015
40,tt4912910,Mission: Impossible - Fallout,Mission: Impossible - Fallout,2018
41,tt9603208,Mission: Impossible - The Final Reckoning,Mission: Impossible - The Final Reckoning,2025
42,tt9603212,Mission: Impossible: Dead Reckoning Partie 1,Mission: Impossible - Dead Reckoning Part One,2023


In [38]:
pattern = "|".join([re.escape(str(x).lower().strip()) for x in clean_movie])

# Filtre les lignes où french_title contient au moins un des termes de clean_movie
act[act["french_title"].str.lower().str.contains(pattern, na=False) | act["original_title"].str.lower().str.contains(pattern, na=False)]

,movie_id,french_title,original_title,year
1,tt1229238,Mission: Impossible - Protocole fantôme,Mission: Impossible - Ghost Protocol,2011
2,tt2381249,Mission: Impossible - Rogue Nation,Mission: Impossible - Rogue Nation,2015


In [29]:
act

,movie_id,french_title,original_title,year
0,tt0083169,Taps,Taps,1981
1,tt0085154,L'esprit d'équipe,All the Right Moves,1983
2,tt0085868,American Teenagers,Losin' It,1983
3,tt0086066,Outsiders,The Outsiders,1983
4,tt0086200,Risky Business,Risky Business,1983
5,tt0089469,Legend,Legend,1985
6,tt0090863,La couleur de l'argent,The Color of Money,1986
7,tt0092099,Top Gun,Top Gun,1986
8,tt0094889,Cocktail,Cocktail,1988
9,tt0095953,Rain Man,Rain Man,1988


In [2]:
def get_soup(url, headers):
    req = Request(url, headers=headers)
    return BeautifulSoup(urlopen(req).read(), "html.parser")

def parse_json_ld(soup):
    script_tag = soup.find('script', {'type': 'application/ld+json'})
    if not script_tag:
        return {}
    return json.loads(script_tag.string)

def get_year(soup):
    for link in soup.find_all('a', {'class': re.compile('ipc-link')}):
        if link.text[:2] in ("19", "20"):
            return int(link.text[:4])
    return None

def get_production_info(soup):
    companies = soup.find_all('a', {'href': re.compile('company')})
    
    if "Production compan" not in companies[0].text:
        prod_names = []
    else :
        prod_names = [c.text for c in companies if "IMDbPro" not in c.text and c.text != "" and "Production compan" not in c.text]
        
    prod_codes = [str(c).split("company/")[1].split("/?ref")[0] for c in companies if "company/" in str(c)]
        
    return prod_names, prod_codes

def parse_duration(duration):
    duration = duration.replace("PT", "")

    hours = 0
    minutes = 0

    if "H" in duration:
        try:
            hours = int(duration.split("H")[0])
            duration = duration.split("H")[1]
        except ValueError:
            hours = 0

    if "M" in duration:
        try:
            minutes = int(duration.split("M")[0])
        except ValueError:
            minutes = 0

    return hours * 60 + minutes

def get_cast_and_director(soup):
    sections = soup.select('[class="ipc-page-section ipc-page-section--base ipc-page-section--bp-none"]')

    dir_section = None
    cast_section = None

    for section in sections:
        head = section.find('div', {"class": re.compile("ipc-title__wrapper")})
        
        if dir_section != None and cast_section != None:
            break
        elif "Director" in [a.text.strip() for a in head][0]:
            dir_section = section
            directors = [d.text.strip() for d in dir_section.find_all('a', {"class": re.compile("title-text-big")})]
            dir_codes = [a['href'].split('/')[2] for a in dir_section.find_all('a', {"class": re.compile("title-text-big")})]
        elif [a.text.strip() for a in head][0] == "Cast":
            cast_section = section
            actors = cast_section.find_all('a', {"class": re.compile("title-text-big")})
            cast_names = [a.text.strip() for a in actors]
            cast_codes = [(str(actors[i]).split('/name/')[1]).split('/?ref')[0] for i in range(len(actors))]

            spans = cast_section.findAll('span')
            good_span = [span for span in spans if span.get_text(strip=True) != "/"]

            cast_voice = []
            for idx, span in enumerate(good_span):
                if "ipc-metadata-list-summary-item__t" not in span.get("class", []):
                    continue

                if idx == len(good_span) - 1:
                    cast_voice.append("A")
                    continue

                next_text = good_span[idx + 1].get_text(strip=True).lower()

                if "(voice)" in next_text:
                    cast_voice.append("V")
                elif "uncredited" in next_text:
                    cast_voice.append("U")
                else:
                    cast_voice.append("A")

            mask = [el != "U" for el in cast_voice]
            cast_names = [cast_names[i] for i in range(len(mask)) if mask[i]]
            cast_codes = [cast_codes[i] for i in range(len(mask)) if mask[i]]
            cast_voice = [cast_voice[i] for i in range(len(mask)) if mask[i]]

            if len(cast_names) > 30:
                cast_names, cast_codes, cast_voice = cast_names[:30], cast_codes[:30], cast_voice[:30]

    return cast_names, cast_codes, cast_voice, directors, dir_codes

def get_budget(soup):
    try:
        spans = soup.find_all('span', {'class': 'ipc-metadata-list-item__list-content-item'})
        budget_text = [s.text for s in spans if "estimé" in s.text][0]
        parts = budget_text.replace("\u202f", "").replace("\xa0", " ").split(" ")
        return int(parts[0]), parts[1]
    except Exception:
        return 0, ""

def movie_imdb(imdb_id):
    HEADERS = {
      'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/99.0.4844.84 Safari/537.36',
      'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.9',
      'Accept-Charset': 'ISO-8859-1,utf-8;q=0.7,*;q=0.3',
      'Accept-Encoding': 'none',
      'Accept-Language': 'fr;q=0.9,en-US,en;q=0.8',
      'Connection': 'keep-alive',
      'refere': 'https://imdb.com'}
    
    # First page
    soup = get_soup(f"https://www.imdb.com/title/{imdb_id}/", headers=HEADERS)
    info = parse_json_ld(soup)
    
    original_title = info.get("name", "")
    rating = info.get("aggregateRating", {}).get("ratingValue", "")
    num_rate = info.get("aggregateRating", {}).get("ratingCount", "")
    year = get_year(soup)
    duration = parse_duration(info.get("duration", ""))
    genres = info.get("genre", [])
    country = [c.text for c in soup.find_all('a', {'href': re.compile('country_of_origin')})] or ["No Info"]
    country = ["USA" if c == "United States" else "UK" if c == "United Kingdom" else c for c in country]
    language = [l.text for l in soup.find_all('a', {'href': re.compile('primary_language')})] or ["No Info"]
    prod_names, prod_codes = get_production_info(soup)

    # Full credits page
    soup2 = get_soup(f"https://www.imdb.com/title/{imdb_id}/fullcredits", headers=HEADERS)
    cast, cast_codes, cast_roles, directors, dir_codes = get_cast_and_director(soup2)

    # Budget (French page)
    HEADERS['Accept-Language'] = 'fr-CH, fr;q=0.9,en-US,en;q=0.8'
    soup3 = get_soup(f"https://www.imdb.com/title/{imdb_id}/", headers=HEADERS)
    info_fr = parse_json_ld(soup3)
    title = info_fr.get("alternateName", info_fr.get("name", ""))
    
    errors = {"&apos;":"'","&amp;":"&"}

    for val in errors.keys():
        original_title = original_title.replace(val, errors[val])
        title = title.replace(val, errors[val])
        
    budget, bud_currency = get_budget(soup3)

    if rating != "":
        return imdb_id, title, original_title, year, directors, dir_codes, cast, cast_codes, cast_roles, genres, duration, \
           country, language, prod_names, prod_codes, rating, num_rate, budget, bud_currency
    else:
        return imdb_id, title,"","","","","","","","","","","","","","","","",""




In [3]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def get_movies_multithread(imdb_ids, max_workers=10):
    results = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(movie_imdb, imdb_id): imdb_id for imdb_id in imdb_ids}

        for future in as_completed(futures):
            imdb_id = futures[future]

            data = future.result()
            results.append(data)

    return results


In [4]:
# import movies to scrap
df = pd.read_csv('scrap_it.csv', encoding="ISO-8859-1", sep=";")
imdb_id = df["imdb"].values

In [5]:
db_df = pd.read_csv('db_backup/movies_db.csv', encoding="ISO-8859-1", sep=";")

In [6]:
imdb_db = db_df["movie_id"].values
final_imdb = [i for i in imdb_id if i not in imdb_db]

In [7]:
final_imdb = list(set(final_imdb))

In [8]:
movies = get_movies_multithread(final_imdb, max_workers=10)

IndexError: list index out of range

In [9]:
movie_df = pd.DataFrame(movies,columns = ["movie_id","french_title","original_title","year","director","director_id",\
                                          "actor","actor_id","status","genre","duration","country",\
                                          "language","production","prod_id","rating","num_rate",\
                                          "budget","currency"])

In [10]:
# keep movies with a rating (the other are probably not available)
movie_keep = movie_df.drop(movie_df[movie_df["year"] == ""].index).reset_index(drop=True)
too_soon = pd.DataFrame(movie_df[["movie_id","french_title"]].drop(movie_df[movie_df["year"] != ""].index).values, \
            columns= ['imdb_id', 'title']).set_index("imdb_id")

In [11]:
# save scraped movies with no ratings
if len(too_soon) > 0:
    f_scrap = pd.read_csv('too_soon.csv', encoding="ISO-8859-1", sep=";").set_index("imdb_id")
    too_soon = pd.concat([f_scrap, too_soon])
    too_soon.to_csv('too_soon.csv', encoding="ISO-8859-1", sep=";")

In [12]:
changer = {"$US":1, "$CA":0.72, "$AU":0.65, "€":1.08, "£GB":1.29, "₩":0.00073, "₹":0.012, "CNY":0.14, "RUR":0.011, \
          "CZK":0.043, "NOK":0.092, "BDT":0.0085, "HKD":0.13, "R$":0.162, "CHF":0.8, "MYR":0.24}

In [13]:
movie_keep["def_budget"] = movie_keep["budget"] * movie_keep["currency"].apply(lambda x : changer[x] if x != "" else 1)

In [14]:
movie_keep["def_budget"] = movie_keep["def_budget"].apply(lambda x: int(x) if x !=0 else "")

In [15]:
modifs = ["actor","country","director","genre","language","production"]

In [16]:
for mod in modifs:
    movie_keep[f"{mod}s"] = movie_keep[mod].apply(lambda x : " | ".join(x))

In [17]:
movie_keep["saw"] = False
movie_keep["wishlist"] = False

In [18]:
select = movie_keep[["movie_id","saw","wishlist","french_title","original_title","year","directors","actors","genres",\
                     "duration","countrys","languages","productions","rating","num_rate","def_budget"]]

In [19]:
select.to_excel("movies_scraped.xlsx", index=False)

In [23]:
select.loc[21,"french_title"] = "ATHOS - Au coeur de la Patrouille de France"

In [24]:
# update movies_db
up_movies = select[["movie_id","french_title","original_title","year","duration","rating","num_rate","def_budget"]]\
                    .set_index("movie_id").rename(columns = {"def_budget":"budget"})
up_movies.to_csv("update_db/up_movies.csv", encoding="ISO-8859-1", sep=";")

In [25]:
# update user_db
up_user = select[["movie_id","saw","wishlist"]].set_index("movie_id")
up_user.to_csv("update_db/up_user.csv", encoding="ISO-8859-1", sep=";")

In [26]:
genre_df = pd.read_csv("db_backup/genres_db.csv", sep=";")

In [27]:
genre_dict = dict(zip(genre_df["name"], genre_df["genre_id"]))
genre_dict[""] = ""

In [29]:
tot_genre_df = movie_keep[["movie_id","genre"]].explode(["genre"], ignore_index=True)
tot_genre_df = tot_genre_df.fillna("")
tot_genre_df["genre_id"] = tot_genre_df["genre"].apply(lambda x: genre_dict[x])
tot_genre_df = tot_genre_df.drop(["genre"], axis=1).drop_duplicates()
tot_genre_df = tot_genre_df[tot_genre_df["genre_id"].notna() & (tot_genre_df["genre_id"].astype(str).str.strip() != "")]

In [30]:
tot_genre_df.set_index("movie_id").to_csv("update_db/up_movie_genre.csv", sep=";")

### Update actors

In [31]:
tot_actor_df = movie_keep[["movie_id","actor", "actor_id", "status"]].explode(["actor", "actor_id", "status"], ignore_index=True)
tot_actor_df["status"] = tot_actor_df["status"].str.replace("A", "", regex=False)
tot_actor_final = tot_actor_df.drop(["actor"], axis=1).drop_duplicates()
tot_actor_final = tot_actor_final[tot_actor_final["actor_id"].notna() & (tot_actor_final["actor_id"].str.strip() != "")]
tot_actor_final.set_index("movie_id").to_csv("update_db/up_movie_actor.csv", sep=";")

In [32]:
act_db = pd.read_csv(f'db_backup/actors_db.csv', encoding="ISO-8859-1", sep=";")

actors_df = tot_actor_df.drop(["movie_id", "status"], axis=1).drop_duplicates()
new_actors = actors_df[~actors_df["actor_id"].isin(act_db["actor_id"])]
new_actors= new_actors[new_actors["actor_id"].notna() & (new_actors["actor_id"].str.strip() != "")]
if len(new_actors) != 0:
    new_actors.columns = ["name","actor_id"]
    new_actors.set_index("actor_id").to_csv("update_db/up_actor.csv", encoding="ISO-8859-1", sep=";")

### Update director and production

In [33]:
db_dict = {"dir":["director","director_id"],"prod":["production","prod_id"]}

In [34]:
for k, v in db_dict.items():
    tot_df = movie_keep[["movie_id",v[0], v[1]]].explode([v[0], v[1]], ignore_index=True)
    tot_df_final = tot_df.drop([v[0]], axis=1).drop_duplicates()
    tot_df_final = tot_df_final[tot_df_final[v[1]].notna() & (tot_df_final[v[1]].str.strip() != "")]
    tot_df_final.set_index("movie_id").to_csv(f"update_db/up_movie_{k}.csv", sep=";")

    val_db = pd.read_csv(f'db_backup/{v[0]}_db.csv', encoding="ISO-8859-1", sep=";")
    
    val_df = tot_df.drop(["movie_id"], axis=1).drop_duplicates()
    new_val = val_df[~val_df[v[1]].isin(val_db[v[1]])]
    if len(new_val) != 0:
        new_val.columns = ["name",v[1]]
        new_val.set_index(v[1]).to_csv(f"update_db/up_{v[0]}.csv", encoding="ISO-8859-1", sep=";")

### Update Country and Language

In [35]:
db_list = ["country", "language"]

In [36]:
def get_or_create_id(name):
    global max_id
    if name not in info_dict.keys():
        max_id += 1
        info_dict[name] = max_id
    return info_dict[name]

In [37]:
for val in db_list:
    val_db2 = pd.read_csv(f'db_backup/{val}_db.csv', encoding="ISO-8859-1", sep=";")
    info_dict = dict(zip(val_db2["name"], val_db2[f"{val}_id"]))
    max_id = max(val_db2[f"{val}_id"])
    
    tot_df2 = movie_keep[["movie_id",val]].explode([val], ignore_index=True)
    tot_df2[f"{val}_id"] = tot_df2[val].apply(get_or_create_id)
    tot_df2.columns = ["movie_id","name",f"{val}_id"]
    tot_df2_final = tot_df2.drop("name", axis=1).drop_duplicates()
    tot_df2_final = tot_df2_final[tot_df2_final[f"{val}_id"].notna() & (tot_df2_final[f"{val}_id"] != "")]
    tot_df2_final.set_index("movie_id").to_csv(f"update_db/up_movie_{val}.csv", sep=";")
    
    val_df2 = tot_df2.drop(["movie_id"], axis=1).drop_duplicates()
    new_val2 = tot_df2[~tot_df2[f"{val}_id"].isin(val_db2[f"{val}_id"])]
    if len(new_val2) != 0:
        new_val2.set_index(f"{val}_id").to_csv(f"update_db/up_{val}.csv", encoding="ISO-8859-1", sep=";")

In [38]:
import sqlalchemy as db
import psycopg2
import pandas as pd
import os

In [39]:
from dotenv import load_dotenv

In [40]:
load_dotenv()

True

In [41]:
DATABASE_URL = os.getenv("DATABASE_URL")

In [42]:
engine = db.create_engine(DATABASE_URL)
connection = engine.connect()

In [43]:
dict_file = {"movies_db":["up_movies", "movie_id", "movies"], "actors_db":["up_actor", "actor_id", "actors"], \
             "country_db":["up_country", "country_id", "country"], "director_db":["up_director", "director_id","directors"], \
             "genres_db":["up_genre", "genre_id", "genres"], "language_db":["up_language", "language_id", "language"], \
             "production_db":["up_production", "prod_id", "production"],"movie_actor_db_2":["up_movie_actor", "movie_id", "movie_actor"], \
             "movie_country_db":["up_movie_country", "movie_id", "movie_country"], "movie_dir_db":["up_movie_dir", "movie_id", "movie_director"], \
             "movie_genre_db":["up_movie_genre", "movie_id", "movie_genre"], "movie_language_db":["up_movie_language", "movie_id", "movie_language"], \
             "movie_prod_db":["up_movie_prod", "movie_id", "movie_prod"], "user_list_db":["up_user", "movie_id", "user_list"]}

In [44]:
for key, value in dict_file.items():
    if os.path.isfile(f"update_db/{value[0]}.csv"):
        database = pd.read_csv(f"db_backup/{key}.csv",encoding="ISO-8859-1", sep=";").set_index(value[1])
        up = pd.read_csv(f"update_db/{value[0]}.csv",encoding="ISO-8859-1", sep=";").set_index(value[1])
        up.to_sql(value[2], engine, if_exists="append")
        pd.concat([database, up]).to_csv(f"db_backup/{key}.csv",encoding="ISO-8859-1", sep=";")